# Modül 08: RNN, LSTM ve GRU
## Deep Learning Path
---
## İçindekiler
1. Öğrenme Hedefleri
2. Vanilla RNN — Matematiksel Temel
3. Vanishing Gradient in RNNs
4. LSTM — 4 Kapı Tam Analizi
5. GRU — Basitleştirilmiş LSTM
6. Bidirectional RNN
7. Seq2Seq Mimarisi
8. TF / PyTorch
9. Alıştırmalar

## 1. Öğrenme Hedefleri
- Vanilla RNN forward pass ve BPTT'yi açıklamak
- RNN'de vanishing gradient problemini kanıtlamak
- LSTM'in 4 kapısını formülle açıklamak
- GRU ile LSTM farkını karşılaştırmak
- Bidirectional RNN mantığını uygulamak
- TF ve PyTorch'ta LSTM modeli kodlamak

## 2. Vanilla RNN Matematiği

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$
$$y_t = W_{yh} \cdot h_t + b_y$$

BPTT (Backprop Through Time): $\frac{\partial L}{\partial h_0} = \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}}$ — her adımda tanh türevi çarpılır → vanishing!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-500,500)))

class LSTMCell:
    def __init__(self, ni, nh):
        n = ni+nh; s=0.1
        self.Wf=np.random.randn(nh,n)*s; self.Wi=np.random.randn(nh,n)*s
        self.Wc=np.random.randn(nh,n)*s; self.Wo=np.random.randn(nh,n)*s
        self.bf=np.zeros((nh,1)); self.bi=np.zeros((nh,1))
        self.bc=np.zeros((nh,1)); self.bo=np.zeros((nh,1))
    def forward(self,x,h,c):
        cat=np.vstack([h,x])
        f=sigmoid(self.Wf@cat+self.bf); i=sigmoid(self.Wi@cat+self.bi)
        ct=np.tanh(self.Wc@cat+self.bc); o=sigmoid(self.Wo@cat+self.bo)
        c_new=f*c+i*ct; h_new=o*np.tanh(c_new)
        return h_new,c_new,(f,i,ct,o)

class GRUCell:
    def __init__(self,ni,nh):
        n=ni+nh; s=0.1
        self.Wr=np.random.randn(nh,n)*s; self.Wz=np.random.randn(nh,n)*s
        self.Wh=np.random.randn(nh,n)*s
        self.br=np.zeros((nh,1)); self.bz=np.zeros((nh,1)); self.bh=np.zeros((nh,1))
    def forward(self,x,h):
        cat=np.vstack([h,x])
        r=sigmoid(self.Wr@cat+self.br); z=sigmoid(self.Wz@cat+self.bz)
        catr=np.vstack([r*h,x]); ht=np.tanh(self.Wh@catr+self.bh)
        return (1-z)*h+z*ht,(r,z)

np.random.seed(0)
lstm=LSTMCell(4,8); gru=GRUCell(4,8)
h=np.zeros((8,1)); c=np.zeros((8,1)); h_g=np.zeros((8,1))
print("LSTM 5 adım:")
for t in range(5):
    x=np.random.randn(4,1)
    h,c,(f,ig,_,o)=lstm.forward(x,h,c)
    print(f"  t={t+1}: ||h||={np.linalg.norm(h):.3f}  f_mean={f.mean():.3f}  i_mean={ig.mean():.3f}")
print("\nGRU 5 adım:")
for t in range(5):
    x=np.random.randn(4,1)
    h_g,(r,z)=gru.forward(x,h_g)
    print(f"  t={t+1}: ||h||={np.linalg.norm(h_g):.3f}  r={r.mean():.3f}  z={z.mean():.3f}")


## 3. Vanishing Gradient — Kanıt

RNN'de BPTT ile gradyan $T$ adım geri gider:
$$\frac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T} \tanh'(h_t) \cdot W_{hh}$$

tanh türevi ≤ 1 ve tipik ~0.5 → 50 adım: $(0.5)^{50} \approx 10^{-15}$

LSTM çözümü: **Cell state** gradyanı kesintisiz taşır (highway).


In [ ]:
steps=np.arange(1,51)
fig,axes=plt.subplots(1,2,figsize=(13,4))

ax1=axes[0]
ax1.semilogy(steps,(0.5)**steps,'r-',lw=2,label='Vanilla RNN (tipik 0.5/adım)')
ax1.semilogy(steps,np.ones(50)*0.9,'b-',lw=2,label='LSTM (cell state)')
ax1.semilogy(steps,(0.85)**steps,'g-',lw=2,label='GRU (update gate)')
ax1.set_title('Uzun Vadeli Gradyan Akışı',fontweight='bold')
ax1.set_xlabel('Geri Adım'); ax1.set_ylabel('Gradyan (log)'); ax1.legend(); ax1.grid(True,alpha=.3)

ax2=axes[1]
hs=[16,32,64,128,256]; ni=32
ax2.plot(hs,[4*(h*(h+ni)+h) for h in hs],'b-o',lw=2,ms=6,label='LSTM (4 kapı)')
ax2.plot(hs,[3*(h*(h+ni)+h) for h in hs],'g-s',lw=2,ms=6,label='GRU (3 kapı)')
ax2.plot(hs,[h*(h+ni)+h for h in hs],'r-^',lw=2,ms=6,label='Vanilla RNN')
ax2.set_title('Parametre Sayısı Karşılaştırması',fontweight='bold')
ax2.set_xlabel('Gizli Boyut'); ax2.set_ylabel('Parametre'); ax2.legend(); ax2.grid(True,alpha=.3)

plt.tight_layout(); plt.show()


## 4. TF / PyTorch Kullanımı

```python
# TensorFlow
tf.keras.layers.LSTM(64, return_sequences=True)
tf.keras.layers.GRU(64, return_sequences=False)
tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64))

# PyTorch
nn.LSTM(input_size=10, hidden_size=64, num_layers=2,
        batch_first=True, dropout=0.3, bidirectional=False)
nn.GRU(10, 64, batch_first=True)
```

**return_sequences=True**: Her adımın çıktısı → (batch, T, hidden)  
**return_sequences=False**: Sadece son adım → (batch, hidden)


## 5. Alıştırmalar

**1.** `LSTMCell` sınıfına backpropagation yaz ve gradyan kontrolü yap.

**2.** Sinüs serisi tahmini: 50 adımlık RNN, LSTM, GRU karşılaştır — hangi daha hızlı yakınsıyor?

**3.** Bidirectional LSTM'i sıfırdan uygula: ileri ve geri geçişlerin çıktısını birleştir.

**4.** Karakter düzeyinde dil modelini gerçekten eğit: Shakespeare metni gibi küçük bir corpus seç.

**5.** `seq2seq` encoder-decoder mimarisini uygula: sayı dizisini tersine çeviren model yaz.

---
**Sonraki Modül:** Modül 09 — Transformer ve Attention
